# Clustering - Complete Concepts Guide
### Using Synthetic Datasets for Clear Understanding

**What you will learn:**
- What is Clustering and why we use it
- K-Means Clustering algorithm step by step
- How to find the best K (Elbow Method + Silhouette Score)
- K-Means Color Quantization
- Hierarchical Clustering and Dendrograms
- DBSCAN - handles noise and irregular shapes
- When to use which algorithm

> **Beginner Tip:** Clustering is **unsupervised learning** — there is no target column (y). The model finds hidden groups on its own!

## 1. Import All Libraries

**New Keywords:**
- `sklearn.cluster` — clustering algorithms (KMeans, AgglomerativeClustering, DBSCAN)
- `sklearn.datasets` — built-in toy datasets
- `sklearn.metrics` — silhouette_score to evaluate clusters
- `scipy.cluster.hierarchy` — dendrogram for hierarchical clustering
- `make_blobs` — creates round, well-separated clusters (perfect for K-Means demo)
- `make_moons`, `make_circles` — creates curved/ring shapes (good for DBSCAN demo)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Clustering algorithms
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN

# Dataset generators
from sklearn.datasets import make_blobs, make_moons, make_circles

# Evaluation
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# Hierarchical dendrogram
from scipy.cluster.hierarchy import dendrogram, linkage

np.random.seed(42)
print("All libraries imported successfully!")

## 2. What is Clustering?

**Clustering** = grouping similar data points together **without any labels**.

### Real-Life Examples:
| Scenario | What we cluster | Groups found |
|---|---|---|
| E-commerce | Customers | High spenders, bargain hunters, window shoppers |
| Medicine | Patients | Risk groups based on symptoms |
| News | Articles | Sports, Politics, Tech, Entertainment |
| Biology | Genes | Similar expression patterns |
| Images | Pixels | Regions of similar color |

### Supervised vs Unsupervised
- **Supervised**: You have labels (y column). Example: Random Forest
- **Unsupervised (Clustering)**: No labels. Model finds structure on its own.

### Three Main Clustering Algorithms:
1. **K-Means** — simple, fast, works for round/spherical clusters
2. **Hierarchical** — builds a tree (dendrogram), no need to pick K beforehand
3. **DBSCAN** — finds arbitrary shapes, handles noise/outliers well

## 3. Generate Synthetic Data for Demo

**New Keywords:**
- `make_blobs` — creates Gaussian (round) clusters
- `n_samples` — how many total data points
- `centers` — how many cluster centers
- `cluster_std` — how spread out each cluster is (standard deviation)
- `random_state` — for reproducibility

In [ ]:
# Create 300 data points with 4 natural clusters
X_blobs, y_true = make_blobs(
    n_samples=300,
    centers=4,
    cluster_std=0.8,
    random_state=42
)

# Convert to DataFrame for easy viewing
df = pd.DataFrame(X_blobs, columns=['Feature_1', 'Feature_2'])
df['True_Label'] = y_true

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head())
print("\nTrue cluster distribution:")
print(df['True_Label'].value_counts().sort_index())

## 4. EDA - Visualize the Data

**Before clustering, always visualize your data!**

**New Keywords:**
- `plt.scatter` — scatter plot (x vs y, each point is one data sample)
- `c=y_true` — color points by their true label
- `cmap='viridis'` — color map (a set of colors)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: data without labels (what clustering sees)
axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], color='gray', alpha=0.6, s=30)
axes[0].set_title('Data Without Labels (What Clustering Sees)')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

# Right: data with true labels (ground truth)
scatter = axes[1].scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_true, cmap='viridis', alpha=0.7, s=30)
axes[1].set_title('True Groups (Ground Truth - Hidden from Model)')
axes[1].set_xlabel('Feature 1')
plt.colorbar(scatter, ax=axes[1])

plt.suptitle('Synthetic Data: 300 Points, 4 Natural Clusters', fontsize=13)
plt.tight_layout()
plt.show()
print("Notice: 4 clear groups visible in the right plot!")

## 5. K-Means Clustering - How It Works

### The Algorithm (Step by Step):

**Step 1:** Choose K (number of clusters) — you decide this!

**Step 2:** Randomly place K centroids (center points) in the data

**Step 3:** Assign each data point to the nearest centroid

**Step 4:** Recalculate centroid = average position of all points in that cluster

**Step 5:** Repeat Steps 3–4 until centroids stop moving (convergence)

### Key Concepts:
- **Centroid** — the "center" of each cluster (mean of all points in it)
- **Inertia (WCSS)** — sum of squared distances from each point to its centroid. Lower = better (tighter clusters)
- **Convergence** — when centroids don't move anymore between iterations

### Inertia Formula:
```
Inertia = Σ Σ ||x_i - μ_k||²
          k  i∈k
```
Where μ_k is the centroid of cluster k and x_i are the points in that cluster.

## 6. K-Means Implementation

**New Keywords:**
- `KMeans(n_clusters=4)` — build a KMeans model with 4 clusters
- `km.fit(X)` — fit (train) the model on data
- `km.labels_` — which cluster each point belongs to (0, 1, 2, ...)
- `km.cluster_centers_` — coordinates of each centroid
- `km.inertia_` — total WCSS (lower = tighter clusters)
- `km.n_iter_` — how many iterations the algorithm ran

In [ ]:
# Build and fit K-Means with K=4
km = KMeans(n_clusters=4, random_state=42, n_init=10)
km.fit(X_blobs)

# Results
labels = km.labels_          # cluster assigned to each point
centers = km.cluster_centers_  # centroid positions
inertia = km.inertia_         # total WCSS

print("Cluster labels (first 10):", labels[:10])
print("Cluster centers shape:", centers.shape)
print("Inertia (WCSS):", round(inertia, 2))
print("Iterations needed:", km.n_iter_)
print("\nPoints per cluster:")
unique, counts = np.unique(labels, return_counts=True)
for c, n in zip(unique, counts):
    print(f"  Cluster {c}: {n} points")

## 7. Visualize K-Means Result

**New Keywords:**
- `km.cluster_centers_` — array of centroid [x, y] coordinates
- Star markers (`marker='*'`) used to show centroids on the plot

In [ ]:
plt.figure(figsize=(8, 5))

# Plot clustered points
plt.scatter(X_blobs[:, 0], X_blobs[:, 1],
            c=labels, cmap='tab10', alpha=0.7, s=30, label='Data Points')

# Plot centroids as big stars
plt.scatter(centers[:, 0], centers[:, 1],
            c='red', marker='*', s=300, zorder=5, label='Centroids')

plt.title('K-Means Clustering Result (K=4)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.tight_layout()
plt.show()
print("Red stars = centroids (cluster centers)")

## 8. Finding Optimal K - The Elbow Method

**Problem:** How do we know what value of K to use?

**Elbow Method:**
- Train K-Means for K = 1, 2, 3, ... 10
- Record inertia (WCSS) for each K
- Plot K vs Inertia → look for the "elbow" (sharp bend)
- The elbow point = optimal K

**Why does inertia always decrease?** More clusters = shorter distances = lower WCSS. But too many clusters = overfitting (each point is its own cluster). We want the K where adding more clusters gives little benefit.

In [ ]:
# Calculate inertia for K = 1 to 10
inertias = []
K_range = range(1, 11)

for k in K_range:
    km_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_temp.fit(X_blobs)
    inertias.append(km_temp.inertia_)

# Plot Elbow Curve
plt.figure(figsize=(8, 4))
plt.plot(K_range, inertias, 'bo-', markersize=8)
plt.axvline(x=4, color='red', linestyle='--', label='Optimal K=4 (elbow)')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia (WCSS)')
plt.title('Elbow Method - Finding Optimal K')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("The elbow is at K=4 — after that, inertia barely decreases!")

## 9. Silhouette Score - Better Validation

**Problem with Elbow Method:** Sometimes the elbow is not obvious!

**Silhouette Score** is a better metric:
- Measures how well each point fits its cluster vs other clusters
- Range: **-1 to +1**
  - +1 = point perfectly belongs to its cluster
  - 0 = point is on the border between two clusters
  - -1 = point is in the wrong cluster

**Formula:**
```
s(i) = (b(i) - a(i)) / max(a(i), b(i))
```
- a(i) = average distance to points in **same** cluster (cohesion)
- b(i) = average distance to points in **nearest other** cluster (separation)

**Higher silhouette score = better clustering!**

In [ ]:
# Calculate silhouette scores for K = 2 to 10
silhouette_scores = []
K_range = range(2, 11)  # silhouette needs at least 2 clusters

for k in K_range:
    km_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_temp = km_temp.fit_predict(X_blobs)
    score = silhouette_score(X_blobs, labels_temp)
    silhouette_scores.append(score)
    print(f"K={k}: Silhouette Score = {score:.4f}")

# Plot
plt.figure(figsize=(8, 4))
plt.plot(K_range, silhouette_scores, 'gs-', markersize=8)
plt.axvline(x=4, color='red', linestyle='--', label='Best K=4')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score vs K (Higher = Better)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. K-Means Color Quantization

**What is Color Quantization?**
Reduce the number of colors in an image while keeping it visually similar. Each unique color = a cluster. K-Means groups similar colors into K representative colors.

**Real Use Cases:**
- Image compression (reduce file size)
- Poster/cartoon effect in photo editing
- Reducing memory in graphics

**How it works:**
- Each pixel has 3 values: (R, G, B) → treat as a point in 3D space
- K-Means finds K color clusters
- Replace each pixel with its centroid color (the cluster's representative color)

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import numpy as np

# Create a colorful synthetic image (100x100 pixels, 3 color channels)
np.random.seed(42)
h, w = 100, 100

# 4 colored quadrants
img = np.zeros((h, w, 3), dtype=np.uint8)
img[:50, :50] = [220, 50, 50]    # top-left: red
img[:50, 50:] = [50, 150, 220]   # top-right: blue
img[50:, :50] = [50, 200, 80]    # bottom-left: green
img[50:, 50:] = [255, 200, 50]   # bottom-right: yellow

# Add some noise so clusters are not trivially perfect
noise = np.random.randint(-30, 30, img.shape)
img = np.clip(img.astype(int) + noise, 0, 255).astype(np.uint8)

print("Image shape:", img.shape)  # (100, 100, 3)
print("Total pixels:", h * w)
print("Unique colors before quantization:", len(np.unique(img.reshape(-1,3), axis=0)))

plt.figure(figsize=(4, 4))
plt.imshow(img)
plt.title("Original Synthetic Image")
plt.axis('off')
plt.show()

In [ ]:
# Flatten image: (100, 100, 3) → (10000, 3)
pixels = img.reshape(-1, 3).astype(float)
print("Pixels array shape:", pixels.shape)

# Apply K-Means with K=4 colors
km_color = KMeans(n_clusters=4, random_state=42, n_init=10)
km_color.fit(pixels)

# Replace each pixel with its centroid color
quantized_pixels = km_color.cluster_centers_[km_color.labels_]
quantized_img = quantized_pixels.reshape(h, w, 3).astype(np.uint8)

print("Unique colors after quantization:", len(np.unique(quantized_img.reshape(-1,3), axis=0)))

# Compare original vs quantized
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(img)
axes[0].set_title("Original (many colors)")
axes[0].axis('off')

axes[1].imshow(quantized_img)
axes[1].set_title("Quantized (K=4 colors only)")
axes[1].axis('off')

plt.suptitle("K-Means Color Quantization", fontsize=13)
plt.tight_layout()
plt.show()
print("Cluster centers (the 4 representative colors):")
print(km_color.cluster_centers_.astype(int))

## 11. K-Means Capstone: Customer Segmentation

**Business Problem:** An e-commerce company wants to segment its customers to send targeted offers.

**Features:**
- `annual_income` — customer's yearly income
- `spending_score` — how much they spend (1-100)
- `age` — customer age

**Goal:** Find natural customer groups to design marketing strategies for each.

In [ ]:
np.random.seed(42)
n = 200

# Create customer segments manually
customer_data = pd.DataFrame({
    'annual_income': np.concatenate([
        np.random.normal(30000, 5000, 50),   # low income
        np.random.normal(70000, 8000, 50),   # mid income
        np.random.normal(120000, 10000, 50), # high income
        np.random.normal(60000, 5000, 50),   # mid income, low spend
    ]),
    'spending_score': np.concatenate([
        np.random.normal(40, 8, 50),   # moderate spenders
        np.random.normal(60, 10, 50),  # active spenders
        np.random.normal(80, 8, 50),   # big spenders
        np.random.normal(20, 6, 50),   # low spenders
    ]),
    'age': np.random.randint(18, 65, 200)
})

# Clip values to realistic ranges
customer_data['annual_income'] = customer_data['annual_income'].clip(10000, 200000)
customer_data['spending_score'] = customer_data['spending_score'].clip(1, 100)

print("Customer Dataset Shape:", customer_data.shape)
print(customer_data.describe().round(1))

In [ ]:
# Scale the features (important for K-Means!)
scaler = StandardScaler()
X_customers = scaler.fit_transform(customer_data)

# Elbow method to find best K
inertias = []
for k in range(1, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_customers)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 3))
plt.plot(range(1, 9), inertias, 'bo-', markersize=8)
plt.xlabel('K')
plt.ylabel('Inertia')
plt.title('Elbow Method - Customer Segmentation')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Apply K-Means with K=4
km_cust = KMeans(n_clusters=4, random_state=42, n_init=10)
customer_data['Segment'] = km_cust.fit_predict(X_customers)

# Segment profiles
segment_profile = customer_data.groupby('Segment')[['annual_income','spending_score','age']].mean().round(0)
print("=== Customer Segment Profiles ===")
print(segment_profile)

# Visualize
plt.figure(figsize=(8, 5))
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
for seg in range(4):
    mask = customer_data['Segment'] == seg
    profile = segment_profile.loc[seg]
    plt.scatter(customer_data.loc[mask, 'annual_income'],
                customer_data.loc[mask, 'spending_score'],
                c=colors[seg], alpha=0.7, s=50,
                label=f'Segment {seg} (Income:{profile.annual_income:.0f}, Score:{profile.spending_score:.0f})')

plt.xlabel('Annual Income')
plt.ylabel('Spending Score')
plt.title('Customer Segments (K-Means K=4)')
plt.legend(fontsize=8, loc='upper left')
plt.tight_layout()
plt.show()

## 12. Hierarchical Clustering - Introduction

### What is Hierarchical Clustering?

Instead of choosing K upfront, hierarchical clustering **builds a tree** of clusters called a **dendrogram**.

**Two Types:**
1. **Agglomerative (Bottom-Up)** ← Most common
   - Start: each point is its own cluster
   - Merge the two closest clusters repeatedly
   - Until only 1 cluster remains
2. **Divisive (Top-Down)**
   - Start: all points in one cluster
   - Split into smaller clusters repeatedly

### Key Concepts:
- **Dendrogram** — the tree diagram showing how clusters merge
- **Linkage** — how to measure distance between clusters:
  - `ward` — minimizes total intra-cluster variance (best default)
  - `complete` — max distance between clusters
  - `average` — average distance between clusters
  - `single` — min distance between clusters

### Advantage over K-Means:
- No need to specify K beforehand — read K from the dendrogram!
- Shows the full hierarchy of relationships
- Better for small to medium datasets

## 13. Dendrogram - Visualizing the Hierarchy

**New Keywords:**
- `linkage(X, method='ward')` — computes the linkage matrix (how clusters merge)
- `dendrogram(Z)` — plots the tree from the linkage matrix
- **Height of merge** — how different two clusters are when they merge
- **Cutting the dendrogram** — draw a horizontal line to decide K

**How to read a dendrogram:**
- Each leaf = one data point
- Each horizontal line = a merge
- Higher line = more dissimilar clusters merged
- Cut where there is the longest vertical line without any horizontal crossings = optimal K

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

# Use a small subset for clear dendrogram (too many points = cluttered)
X_sample = X_blobs[:30]

# Compute linkage matrix
Z = linkage(X_sample, method='ward')

# Plot dendrogram
plt.figure(figsize=(12, 5))
dendrogram(Z,
           leaf_rotation=90,
           leaf_font_size=8,
           color_threshold=10)
plt.axhline(y=10, color='red', linestyle='--', label='Cut here → K=4')
plt.title('Dendrogram (Ward Linkage) - 30 Sample Points')
plt.xlabel('Data Point Index')
plt.ylabel('Distance (Dissimilarity)')
plt.legend()
plt.tight_layout()
plt.show()
print("Red dashed line = cut level. Count branches below = number of clusters.")

## 14. Hierarchical Clustering Implementation

**New Keywords:**
- `AgglomerativeClustering` — sklearn's implementation of bottom-up clustering
- `n_clusters=4` — how many clusters (cut level)
- `linkage='ward'` — ward linkage (minimizes variance)
- `metric='euclidean'` — distance measure between points

In [ ]:
from sklearn.cluster import AgglomerativeClustering

# Hierarchical Clustering
hc = AgglomerativeClustering(
    n_clusters=4,
    linkage='ward',    # ward minimizes within-cluster variance
    metric='euclidean'
)
hc_labels = hc.fit_predict(X_blobs)

# Silhouette score
hc_sil = silhouette_score(X_blobs, hc_labels)
km_sil = silhouette_score(X_blobs, labels)  # from K-Means earlier

print(f"K-Means Silhouette Score:       {km_sil:.4f}")
print(f"Hierarchical Silhouette Score:  {hc_sil:.4f}")

# Compare K-Means vs Hierarchical
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(X_blobs[:,0], X_blobs[:,1], c=labels, cmap='tab10', s=30, alpha=0.7)
axes[0].set_title(f'K-Means (Silhouette={km_sil:.3f})')

axes[1].scatter(X_blobs[:,0], X_blobs[:,1], c=hc_labels, cmap='tab10', s=30, alpha=0.7)
axes[1].set_title(f'Hierarchical Clustering (Silhouette={hc_sil:.3f})')

for ax in axes:
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.suptitle('K-Means vs Hierarchical Clustering', fontsize=13)
plt.tight_layout()
plt.show()

## 15. Hierarchical Clustering Capstone - Document Grouping

**Business Problem:** A company has employee survey responses (numerical scores). Group employees into similar profile groups for HR insights.

**Features:**
- `work_life_balance` — score 1-10
- `job_satisfaction` — score 1-10
- `performance_rating` — score 1-5
- `years_at_company` — integer

In [ ]:
np.random.seed(42)
n_emp = 100

employee_df = pd.DataFrame({
    'work_life_balance': np.concatenate([
        np.random.normal(8, 0.8, 25),   # group A: high WLB
        np.random.normal(4, 0.8, 25),   # group B: low WLB
        np.random.normal(7, 0.8, 25),   # group C: moderate WLB
        np.random.normal(5, 0.8, 25),   # group D: moderate-low WLB
    ]),
    'job_satisfaction': np.concatenate([
        np.random.normal(8, 0.8, 25),
        np.random.normal(3, 0.8, 25),
        np.random.normal(6, 0.8, 25),
        np.random.normal(7, 0.8, 25),
    ]),
    'performance_rating': np.concatenate([
        np.random.normal(4.5, 0.3, 25),
        np.random.normal(2.5, 0.4, 25),
        np.random.normal(3.5, 0.3, 25),
        np.random.normal(4.0, 0.3, 25),
    ]),
    'years_at_company': np.concatenate([
        np.random.randint(5, 20, 25),
        np.random.randint(1, 5, 25),
        np.random.randint(3, 15, 25),
        np.random.randint(2, 10, 25),
    ]).astype(float)
})
# Clip scores to valid ranges
for col in ['work_life_balance', 'job_satisfaction']:
    employee_df[col] = employee_df[col].clip(1, 10)
employee_df['performance_rating'] = employee_df['performance_rating'].clip(1, 5)

print("Employee Dataset Shape:", employee_df.shape)
print(employee_df.head())

In [ ]:
# Scale and cluster
scaler_emp = StandardScaler()
X_emp = scaler_emp.fit_transform(employee_df)

# Dendrogram
Z_emp = linkage(X_emp, method='ward')
plt.figure(figsize=(14, 4))
dendrogram(Z_emp, leaf_rotation=90, leaf_font_size=6, color_threshold=4)
plt.axhline(y=4, color='red', linestyle='--', label='Cut: K=4')
plt.title('Employee Survey Dendrogram')
plt.xlabel('Employee Index')
plt.ylabel('Distance')
plt.legend()
plt.tight_layout()
plt.show()

# Apply hierarchical clustering
hc_emp = AgglomerativeClustering(n_clusters=4, linkage='ward')
employee_df['Group'] = hc_emp.fit_predict(X_emp)

# Profile each group
print("\n=== Employee Group Profiles ===")
print(employee_df.groupby('Group').mean().round(2))

## 16. DBSCAN - Introduction

### What is DBSCAN?
**DBSCAN = Density-Based Spatial Clustering of Applications with Noise**

Unlike K-Means and Hierarchical, DBSCAN:
- **Does NOT need K** — finds the number of clusters automatically
- **Handles arbitrary shapes** — not just round blobs
- **Identifies outliers/noise** — labels them as -1
- **Works on density** — areas with many points = cluster

### Three Types of Points:
1. **Core Point** — has at least `min_samples` points within radius `eps`
2. **Border Point** — within `eps` of a core point but not itself a core point
3. **Noise Point (Outlier)** — not within `eps` of any core point → labeled **-1**

### Key Hyperparameters:
- **eps (epsilon)** — the radius to look for neighbors (neighborhood size)
- **min_samples** — minimum points needed within eps to form a core point

### Why is DBSCAN better than K-Means for some data?
K-Means assumes spherical clusters. DBSCAN can find any shape!

## 17. DBSCAN vs K-Means - When K-Means Fails

**Non-spherical data breaks K-Means!**

Let's create two datasets where K-Means fails but DBSCAN succeeds:
- `make_moons` — two interleaved half-circles
- `make_circles` — ring inside a ring

**New Keywords:**
- `make_moons` — creates two moon/crescent shaped clusters
- `make_circles` — creates inner circle and outer ring
- `noise=0.1` — adds random noise to data points

In [ ]:
# Create challenging datasets
X_moons, y_moons = make_moons(n_samples=300, noise=0.1, random_state=42)
X_circles, y_circles = make_circles(n_samples=300, noise=0.08, factor=0.5, random_state=42)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# K-Means on moons
km_moons = KMeans(n_clusters=2, random_state=42, n_init=10)
axes[0,0].scatter(X_moons[:,0], X_moons[:,1], c=km_moons.fit_predict(X_moons), cmap='tab10', s=20)
axes[0,0].set_title('K-Means on Moons (WRONG!)')

# DBSCAN on moons
db_moons = DBSCAN(eps=0.3, min_samples=5)
axes[0,1].scatter(X_moons[:,0], X_moons[:,1], c=db_moons.fit_predict(X_moons), cmap='tab10', s=20)
axes[0,1].set_title('DBSCAN on Moons (CORRECT!)')

# K-Means on circles
km_circles = KMeans(n_clusters=2, random_state=42, n_init=10)
axes[1,0].scatter(X_circles[:,0], X_circles[:,1], c=km_circles.fit_predict(X_circles), cmap='tab10', s=20)
axes[1,0].set_title('K-Means on Circles (WRONG!)')

# DBSCAN on circles
db_circles = DBSCAN(eps=0.2, min_samples=5)
axes[1,1].scatter(X_circles[:,0], X_circles[:,1], c=db_circles.fit_predict(X_circles), cmap='tab10', s=20)
axes[1,1].set_title('DBSCAN on Circles (CORRECT!)')

for ax in axes.flat:
    ax.set_xlabel('X')
    ax.set_ylabel('Y')

plt.suptitle('K-Means vs DBSCAN on Non-Spherical Data', fontsize=13)
plt.tight_layout()
plt.show()

## 18. DBSCAN Hyperparameters

### eps (epsilon) — Neighborhood Radius
- **Too small**: Most points become noise (too strict)
- **Too large**: All points merge into one cluster (too loose)
- **Just right**: Clear clusters form

### min_samples — Core Point Threshold
- **Too small** (1-2): Everything is a core point, too sensitive
- **Too large**: Many points become noise/border
- **Rule of thumb**: `min_samples >= 2 * n_features` or at least 4-5

**Let's see how changing eps affects results:**

In [ ]:
X_test, _ = make_moons(n_samples=300, noise=0.1, random_state=42)

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
eps_values = [0.1, 0.3, 0.6, 1.5]

for ax, eps_val in zip(axes, eps_values):
    db = DBSCAN(eps=eps_val, min_samples=5)
    lbls = db.fit_predict(X_test)
    n_clusters = len(set(lbls)) - (1 if -1 in lbls else 0)
    n_noise = (lbls == -1).sum()
    ax.scatter(X_test[:,0], X_test[:,1], c=lbls, cmap='tab10', s=15)
    ax.set_title(f'eps={eps_val}\n{n_clusters} clusters, {n_noise} noise')
    ax.set_xlabel('X')

plt.suptitle('Effect of eps on DBSCAN Results', fontsize=12)
plt.tight_layout()
plt.show()
print("eps=0.3 is correct here. Too small = too many noise. Too large = one big cluster.")

## 19. DBSCAN Hyperparameter Tuning - K-Distance Graph

### How to find the best eps?

**K-Distance Graph Method:**
1. For each point, find its distance to the **kth nearest neighbor** (k = min_samples - 1)
2. Sort these distances in descending order and plot them
3. Look for the **"elbow"** in the graph — that's the best eps!

**Intuition:** Points in dense regions have small k-distance. The elbow separates dense regions from sparse ones/noise.

In [ ]:
from sklearn.neighbors import NearestNeighbors

# k-distance graph to find best eps
# k = min_samples - 1 = 4 (we'll use min_samples=5)
k = 4
nbrs = NearestNeighbors(n_neighbors=k).fit(X_moons)
distances, _ = nbrs.kneighbors(X_moons)

# Get kth nearest neighbor distance for each point
k_distances = np.sort(distances[:, -1])[::-1]  # sort descending

plt.figure(figsize=(8, 4))
plt.plot(k_distances, 'b-')
plt.axhline(y=0.3, color='red', linestyle='--', label='Optimal eps ≈ 0.3 (elbow)')
plt.xlabel('Points (sorted by distance)')
plt.ylabel(f'{k}-NN Distance')
plt.title('K-Distance Graph - Finding Optimal eps')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("The 'knee' or elbow in this graph = best eps value!")

In [ ]:
# Apply DBSCAN with the eps found from k-distance graph
db_tuned = DBSCAN(eps=0.3, min_samples=5)
db_labels = db_tuned.fit_predict(X_moons)

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = (db_labels == -1).sum()

print(f"Number of clusters found: {n_clusters}")
print(f"Number of noise points:   {n_noise}")
print(f"Cluster labels: {set(db_labels)}")

# Visualize (noise points = -1 → shown in black)
colors = ['tab:blue' if l != -1 else 'black' for l in db_labels]
plt.figure(figsize=(8, 5))
plt.scatter(X_moons[:,0], X_moons[:,1],
            c=db_labels, cmap='tab10', s=30, alpha=0.8)
plt.title(f'DBSCAN Result: {n_clusters} clusters, {n_noise} noise points (black)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.tight_layout()
plt.show()

## 20. DBSCAN Capstone - Anomaly/Outlier Detection

**Business Problem:** A GPS company wants to detect unusual movement patterns from location data. Normal movement = clusters. Unusual/erratic movement = noise (potential fraud or error).

**DBSCAN is perfect for this because:**
- Normal routes form dense clusters → labeled 0, 1, 2...
- Unusual/isolated points = **noise → labeled -1** (anomaly!)

In [ ]:
np.random.seed(42)

# Normal GPS routes (3 route clusters)
route1 = np.random.normal([2, 2], 0.3, (80, 2))
route2 = np.random.normal([6, 6], 0.3, (80, 2))
route3 = np.random.normal([2, 7], 0.3, (80, 2))

# Anomalous points (random scattered locations)
anomalies = np.random.uniform(0, 9, (15, 2))

X_gps = np.vstack([route1, route2, route3, anomalies])
print("Total GPS points:", X_gps.shape[0])
print("Normal route points: 240")
print("Anomalous points: 15")

In [ ]:
# Apply DBSCAN
db_gps = DBSCAN(eps=0.8, min_samples=5)
gps_labels = db_gps.fit_predict(X_gps)

n_clusters = len(set(gps_labels)) - (1 if -1 in gps_labels else 0)
n_anomalies = (gps_labels == -1).sum()

print(f"Routes (clusters) found: {n_clusters}")
print(f"Anomalies detected: {n_anomalies}")

# Visualize
plt.figure(figsize=(8, 6))
# Normal route points
mask_normal = gps_labels != -1
plt.scatter(X_gps[mask_normal, 0], X_gps[mask_normal, 1],
            c=gps_labels[mask_normal], cmap='tab10',
            s=40, alpha=0.7, label='Normal Route Points')

# Anomalies
mask_anomaly = gps_labels == -1
plt.scatter(X_gps[mask_anomaly, 0], X_gps[mask_anomaly, 1],
            c='red', marker='X', s=100, zorder=5, label=f'Anomalies ({n_anomalies})')

plt.title(f'DBSCAN Anomaly Detection: {n_clusters} Routes, {n_anomalies} Anomalies')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.legend()
plt.tight_layout()
plt.show()
print("Red X marks = anomalous GPS points detected by DBSCAN!")

## 21. Algorithm Comparison Summary

| Feature | K-Means | Hierarchical | DBSCAN |
|---|---|---|---|
| Need to specify K? | YES | NO (or yes) | NO |
| Cluster shape | Spherical only | Any | Any |
| Handles noise/outliers | No | No | YES (-1 label) |
| Speed | Fast | Slow (O(n²)) | Medium |
| Best for | Large data, known K | Small data, hierarchy | Noisy data, odd shapes |
| Memory | Low | High | Medium |
| Sensitive to scale? | YES | YES | YES |

### Quick Decision Guide:
- **Data is well-separated blobs + you know K** → K-Means
- **Want hierarchy / no K known + small data** → Hierarchical
- **Data has noise or non-round shapes** → DBSCAN
- **Always scale** your features before any clustering algorithm!